# NeuralAtmosphereOperator — khám phá và thiết kế dữ liệu khí quyển

> **Phạm vi của notebook.** Tài liệu này chỉ nói về **dữ liệu**: nguồn ERA5/WeatherBench2, ý nghĩa của lưới latitude–longitude, contract cố định 71 kênh, cách giảm độ phân giải từ $0.25^\circ$ xuống $0.5^\circ$, tensor cuối cùng và các kiểm tra cần thực hiện trước khi train. Kiến trúc model, loss, normalization, training pipeline và evaluation không thuộc phạm vi ở đây.

Mục tiêu là trả lời rõ bốn câu hỏi:

1. Mỗi sample mô tả trạng thái khí quyển nào?
2. 71 channels gồm những đại lượng vật lý nào?
3. Lưới $361\times720$ được tạo từ lưới $721\times1440$ như thế nào?
4. Làm sao biết file tải về đúng trước khi dùng nó cho một lần train tốn kém?

> **Quy ước thuật ngữ.** Các thuật ngữ quen thuộc như *channel*, *grid*, *reanalysis*, *pressure level*, *regridding*, *aliasing* và *Zarr* được giữ bằng tiếng Anh khi dịch sang tiếng Việt làm câu văn gượng ép. Mỗi khái niệm đều được giải thích khi xuất hiện lần đầu.

## 1. Dữ liệu đến từ đâu?

Nguồn của dự án là bộ **ERA5** đã được WeatherBench2 chuẩn bị sẵn trên Google Cloud Storage:

> gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr

Tên của dataset cho biết các đặc điểm chính:

- **1959–2023:** khoảng thời gian mà archive nguồn cung cấp; dự án chỉ chọn một đoạn con cần thiết.
- **0.25°:** khoảng cách góc giữa hai grid points kề nhau ở dataset nguồn.
- **wb13:** dữ liệu trên 13 pressure levels chuẩn của WeatherBench.
- **6h:** một trạng thái khí quyển sau mỗi 6 giờ.
- **1440 × 721:** 1440 kinh độ và 721 vĩ độ, bao gồm cả hai cực.
- **with derived variables:** ngoài các trường ERA5 cơ bản còn có một số biến được WeatherBench2 suy ra trước.

ERA5 là **reanalysis**, không phải một tập ảnh vệ tinh thô và cũng không phải dự báo của model khác. Reanalysis kết hợp mô hình vật lý, quan trắc và data assimilation để tái dựng một trạng thái khí quyển nhất quán theo thời gian. Vì vậy mỗi timestep có thể được xem như một ước lượng tốt của trạng thái thật, nhưng không phải phép đo hoàn hảo ở mọi grid point.

Nguồn tham khảo: [WeatherBench2 Data Guide](https://weatherbench2.readthedocs.io/en/latest/data-guide.html), [ERA5 — Hersbach et al., 2020](https://doi.org/10.1002/qj.3803), [WeatherBench2 — Rasp et al., 2024](https://arxiv.org/abs/2308.15560).

## 2. Một timestep trông như thế nào?

Tại một thời điểm $t$, dữ liệu cuối cùng là tensor

$$
\mathbf{x}_t\in\mathbb{R}^{71\times361\times720}.
$$

Ba trục lần lượt là:

- **channel:** 71 trường vật lý được liệt kê ở các phần sau;
- **latitude:** 361 hàng từ $90^\circ$ Bắc đến $90^\circ$ Nam, bước $0.5^\circ$;
- **longitude:** 720 cột từ $0^\circ$ đến $359.5^\circ$, bước $0.5^\circ$.

Dữ liệu có cadence 6 giờ, tương ứng với các mốc 00, 06, 12 và 18 UTC mỗi ngày. Config hiện tại cũng dùng stride 6 giờ, vì vậy **không bỏ bớt timestep** và không nội suy theo thời gian.

Có thể hình dung một timestep như 71 bản đồ toàn cầu xếp chồng lên nhau. Mọi bản đồ dùng chung grid và timestamp, nhưng giá trị cùng đơn vị vật lý của riêng channel đó.

## 3. Đọc lưới latitude–longitude đúng cách

Latitude $\varphi$ đo vị trí Bắc–Nam, longitude $\lambda$ đo vị trí Đông–Tây. Trên grid $0.5^\circ$:

- khoảng cách Bắc–Nam giữa hai hàng xấp xỉ $55.6$ km;
- khoảng cách Đông–Tây phụ thuộc latitude và xấp xỉ $55.6\cos\varphi$ km.

Do đó grid latitude–longitude **không phải equal-area grid**. Một ô gần xích đạo có diện tích lớn hơn nhiều so với một ô gần cực. Với bán kính Trái Đất $R$, diện tích ô có biên $[\varphi_1,\varphi_2]$ và $[\lambda_1,\lambda_2]$ là

$$
A=R^2(\lambda_2-\lambda_1)\left(\sin\varphi_2-\sin\varphi_1\right),
$$

trong đó các góc được tính bằng radian. Hệ số $\sin\varphi$ này là lý do phép regridding và các phép trung bình toàn cầu không được coi mọi grid point có trọng số bằng nhau.

Longitude là trục **periodic**: sau $359.5^\circ$ là $0^\circ$, không có một đường biên vật lý tại kinh tuyến gốc. Latitude thì không periodic; $90^\circ$ Bắc và $90^\circ$ Nam là hai cực.

## 4. Pressure level là gì?

Khí quyển ba chiều thường được biểu diễn trên các mặt có cùng áp suất thay vì các mặt có cùng độ cao. Đơn vị thường dùng là hectopascal:

$$1\ \mathrm{hPa}=100\ \mathrm{Pa}. $$

Áp suất càng thấp thì mặt đó nhìn chung càng cao trong khí quyển. Tuy nhiên pressure level **không phải một độ cao cố định**: độ cao hình học của mặt 500 hPa thay đổi theo nhiệt độ và trạng thái khí quyển.

| Pressure level | Cách hiểu gần đúng | Vai trò thường gặp |
|---:|---|---|
| 1000 hPa | sát mặt đất ở nơi địa hình thấp | gió, ẩm và geopotential tầng thấp |
| 850 hPa | lower troposphere | khối khí, vận chuyển nhiệt và hơi nước |
| 500 hPa | mid-troposphere | rãnh, sống khí áp và circulation quy mô lớn |
| 250 hPa | upper troposphere | jet stream và dòng dẫn hướng |
| 100 hPa | gần tropopause/lower stratosphere | cấu trúc nhiệt tầng cao |
| 50 hPa | lower stratosphere | circulation tầng bình lưu |

Bảng chỉ nhằm tạo trực giác. Quan hệ pressure–height thay đổi theo không gian và thời gian, nên không nên gán cho mỗi pressure level một độ cao tuyệt đối.

> **Lưu ý ở vùng núi cao.** Một pressure surface như 1000 hPa có thể nằm thấp hơn mặt đất địa phương. Giá trị pressure-level của reanalysis tại những vị trí đó chịu ảnh hưởng của cách mô hình/data assimilation biểu diễn hoặc ngoại suy vùng dưới bề mặt; không nên diễn giải nó như một phép đo không khí thật ở dưới địa hình.

# Phần I — 71 channels được đưa vào model

## 5. Sáu surface channels

Surface channel là trường hai chiều, không có trục pressure level trong archive. Sáu trường cố định là:

| Channel | Tên trong WeatherBench2 | Đơn vị | Ý nghĩa |
|---:|---|---|---|
| 1 | 10m_u_component_of_wind | $\mathrm{m\,s^{-1}}$ | thành phần gió hướng Đông–Tây ở 10 m; dương theo hướng Đông |
| 2 | 10m_v_component_of_wind | $\mathrm{m\,s^{-1}}$ | thành phần gió hướng Bắc–Nam ở 10 m; dương theo hướng Bắc |
| 3 | 2m_temperature | K | nhiệt độ không khí ở 2 m |
| 4 | surface_pressure | Pa | áp suất thật tại bề mặt địa hình |
| 5 | mean_sea_level_pressure | Pa | áp suất đã quy về mực nước biển, hữu ích để nhận biết hệ thống synoptic |
| 6 | total_column_water_vapour | $\mathrm{kg\,m^{-2}}$ | tổng lượng hơi nước tích phân theo cột khí quyển |

Hai thành phần gió không phải speed và direction. Tốc độ gió 10 m có thể suy ra bằng

$$
V_{10}=\sqrt{u_{10}^2+v_{10}^2}.
$$

Surface pressure và mean sea-level pressure cũng không trùng nhau: trường thứ nhất chịu ảnh hưởng trực tiếp của độ cao địa hình; trường thứ hai thuận tiện hơn khi so sánh các hệ áp suất trên những vùng có độ cao khác nhau.

## 6. Sáu mươi lăm pressure-level channels

Mỗi cặp variable @ pressure level là một kênh. Cấu hình mới dùng đủ 13 tầng
cho cả năm biến cốt lõi: 50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000 hPa.

| Channels (1-based) | Variable | Số kênh | Ý nghĩa |
|---|---|---:|---|
| 7–19 | u_component_of_wind | 13 | gió ngang Đông–Tây |
| 20–32 | v_component_of_wind | 13 | gió ngang Bắc–Nam |
| 33–45 | geopotential | 13 | trường thế năng trọng trường |
| 46–58 | temperature | 13 | cấu trúc nhiệt động lực học |
| 59–71 | specific_humidity | 13 | phân bố hơi nước |

Không còn lấy thưa tầng như bộ compact 26 kênh ban đầu. Đây là toàn bộ phần
pressure-level của bộ 73 kênh SFNO tham chiếu.


## 7. Channel order chính xác

Thứ tự dưới đây là data contract dùng chung bởi downloader, loader, statistics
và model. Các nhóm pressure-level đi theo thứ tự u, v, z, T, q; mỗi nhóm tăng
dần từ 50 đến 1000 hPa.

| Index (1-based) | Channel |
|---:|---|
| 1 | 10m_u_component_of_wind |
| 2 | 10m_v_component_of_wind |
| 3 | 2m_temperature |
| 4 | surface_pressure |
| 5 | mean_sea_level_pressure |
| 6 | total_column_water_vapour |
| 7 | u_component_of_wind@50hPa |
| 8 | u_component_of_wind@100hPa |
| 9 | u_component_of_wind@150hPa |
| 10 | u_component_of_wind@200hPa |
| 11 | u_component_of_wind@250hPa |
| 12 | u_component_of_wind@300hPa |
| 13 | u_component_of_wind@400hPa |
| 14 | u_component_of_wind@500hPa |
| 15 | u_component_of_wind@600hPa |
| 16 | u_component_of_wind@700hPa |
| 17 | u_component_of_wind@850hPa |
| 18 | u_component_of_wind@925hPa |
| 19 | u_component_of_wind@1000hPa |
| 20 | v_component_of_wind@50hPa |
| 21 | v_component_of_wind@100hPa |
| 22 | v_component_of_wind@150hPa |
| 23 | v_component_of_wind@200hPa |
| 24 | v_component_of_wind@250hPa |
| 25 | v_component_of_wind@300hPa |
| 26 | v_component_of_wind@400hPa |
| 27 | v_component_of_wind@500hPa |
| 28 | v_component_of_wind@600hPa |
| 29 | v_component_of_wind@700hPa |
| 30 | v_component_of_wind@850hPa |
| 31 | v_component_of_wind@925hPa |
| 32 | v_component_of_wind@1000hPa |
| 33 | geopotential@50hPa |
| 34 | geopotential@100hPa |
| 35 | geopotential@150hPa |
| 36 | geopotential@200hPa |
| 37 | geopotential@250hPa |
| 38 | geopotential@300hPa |
| 39 | geopotential@400hPa |
| 40 | geopotential@500hPa |
| 41 | geopotential@600hPa |
| 42 | geopotential@700hPa |
| 43 | geopotential@850hPa |
| 44 | geopotential@925hPa |
| 45 | geopotential@1000hPa |
| 46 | temperature@50hPa |
| 47 | temperature@100hPa |
| 48 | temperature@150hPa |
| 49 | temperature@200hPa |
| 50 | temperature@250hPa |
| 51 | temperature@300hPa |
| 52 | temperature@400hPa |
| 53 | temperature@500hPa |
| 54 | temperature@600hPa |
| 55 | temperature@700hPa |
| 56 | temperature@850hPa |
| 57 | temperature@925hPa |
| 58 | temperature@1000hPa |
| 59 | specific_humidity@50hPa |
| 60 | specific_humidity@100hPa |
| 61 | specific_humidity@150hPa |
| 62 | specific_humidity@200hPa |
| 63 | specific_humidity@250hPa |
| 64 | specific_humidity@300hPa |
| 65 | specific_humidity@400hPa |
| 66 | specific_humidity@500hPa |
| 67 | specific_humidity@600hPa |
| 68 | specific_humidity@700hPa |
| 69 | specific_humidity@850hPa |
| 70 | specific_humidity@925hPa |
| 71 | specific_humidity@1000hPa |

Output lưu channel name, units, long name, source variable và pressure level.


## 8. “Tổng số channels” trong archive là bao nhiêu?

Cần phân biệt **variable** và **channel**. Một variable hai chiều như 2m_temperature tạo một channel. Một variable bốn chiều như temperature có thêm trục pressure level; nếu lấy đủ 13 levels thì riêng variable đó tạo 13 channels.

Metadata của đúng WeatherBench2 archive mà config đang trỏ tới có cấu trúc sau, sau khi bỏ bốn coordinate arrays time, level, latitude và longitude:

| Nhóm | Số variables | Số maps sau khi trải level | Có thay đổi theo thời gian? |
|---|---:|---:|---|
| trường bề mặt hoặc trường đơn tầng | 34 | 34 | có |
| trường trên 13 pressure levels | 15 | $15\times13=195$ | có |
| trường static như địa hình, đất–biển, vegetation | 13 | 13 | không |
| **Tổng** | **62** | **242** | hỗn hợp |

Nếu chỉ đếm các trường động có thể làm forecast state/target thì archive có tối đa

$$
34+15\times13=229\text{ channels}.
$$

Mười ba static fields có thể là auxiliary inputs nhưng không phải các target
tiến hóa theo thời gian. Không phải cả 229 dynamic maps đều có semantics phù
hợp với autoregressive atmospheric state.

**Đã kiểm tra metadata và coordinate trực tiếp ngày 2026-09-09:** bộ SFNO tham
chiếu có 73 kênh, nhưng WB2 này thiếu u/v ở độ cao 100 m. Vì vậy dự án dùng
**71 = 6 + 5 × 13** kênh có sẵn, không thêm biến khác để làm tròn thành 73.
Xem [bằng chứng và đối chiếu nguồn](../docs/DATA_CHANNELS.md).


## 8.1. Lịch sử quyết định: từ compact 26 kênh đến contract 71 kênh hiện hành

Thiết kế ban đầu lấy thưa các tầng để giảm storage và chi phí thử nghiệm.
Thiết kế mới giữ sáu biến bề mặt và mở đủ 13 tầng cho u, v, z, T, q. Mục tiêu là
giữ cấu trúc theo chiều thẳng đứng và tăng thông tin về tương tác giữa các biến.
Relative humidity 500 hPa từng là một biến dẫn xuất trong bộ 26 kênh, không nằm
trong bộ 73 kênh SFNO đang đối chiếu.

Việc mở rộng này không có nghĩa khí quyển có đúng 71 degrees of freedom.
Các biến có tương quan và mô hình vẫn thiếu forcing/coupling. Không thêm wind
speed như một target chỉ để tăng số kênh khi đã có $u,v$ với
$V=\sqrt{u^2+v^2}$.

Các precipitation/flux là tích lũy hoặc trung bình theo thời gian, cần target
window riêng. Các biến land/ocean và static fields cần thiết kế coupling hoặc
auxiliary inputs riêng trước khi đưa vào pipeline này.

Dung lượng tensor 71 kênh float32 ở 0.5° cho 1995–2020 là khoảng **2.804 TB**.
Nếu lưu cả 229 dynamic maps thì khoảng **9.044 TB** chưa nén. Compression và
chunk layout quyết định dung lượng vật lý và network traffic thực tế.


## 8.2. Những gì vẫn chưa đưa vào state và giới hạn

| Nhóm | Ví dụ | Lý do cần thiết kế riêng |
|---|---|---|
| derived dynamics | wind speed, vorticity, divergence | thông tin dẫn xuất từ các trường cơ sở |
| precipitation/flux | precipitation 6/12/24 h, radiation, heat fluxes | semantics theo time window |
| land/ocean | SST, soil moisture, sea ice | coupling và time scales khác |
| static fields | địa hình, land–sea mask | auxiliary inputs, không phải target động |
| 100-m winds của SFNO | u100m, v100m | không tồn tại trong archive này |

Đã giữ đủ 13 tầng áp suất cho năm biến cốt lõi, nhưng chỉ có 13 tầng này không
đảm bảo phân giải mọi cấu trúc thẳng đứng. State 71 kênh vẫn không phải closed
physical system. Chất lượng phải được đánh giá trên dữ liệu held-out.


## 9. Contract 71 kênh có nghĩa gì đối với dữ liệu tải?

Downloader mở Zarr lazy, giới hạn time range và luôn đọc sáu surface variables cùng năm
pressure-level variables trên **đủ 13 tầng**, rồi regrid và ghi state 71 kênh. Không có option chọn bỏ biến hoặc tầng.

Metadata nguồn cho thấy mỗi chunk pressure variable chứa một timestep, đủ
13 tầng và cả grid 721×1440. Vì vậy bộ cũ lấy ít tầng vẫn phải đọc cả chunk
nguồn của biến đó. Mở đủ tầng làm output lớn hơn nhưng không nhất thiết tăng
network traffic theo tỷ lệ 71/26. Chunking, compression và số source variables
được đọc mới quyết định lượng byte qua mạng.

Cần phân biệt state contract, network traffic và output storage. Các data variable
ngoài định nghĩa state SFNO không được trộn vào target tự hồi quy.


# Phần II — giảm grid từ 0.25° xuống 0.5°

## 10. Kích thước grid trước và sau regridding

| Grid | Latitude | Longitude | Số grid points |
|---|---:|---:|---:|
| nguồn $0.25^\circ$ | 721 | 1440 | 1,038,240 |
| đích $0.5^\circ$ | 361 | 720 | 259,920 |

Số grid points giảm theo tỉ lệ

$$
\frac{721\times1440}{361\times720}\approx3.994.
$$

Tức là gần 4 lần, vì độ phân giải giảm 2 lần trên cả hai trục. Tỉ lệ không đúng 4 tuyệt đối do cả grid nguồn và grid đích đều giữ hai hàng ở hai cực.

Regridding chỉ giảm **spatial resolution**. Nó không bỏ timestep và không loại channel nào khỏi contract cố định 71 kênh. Dẫu vậy, chi tiết có wavelength nhỏ hơn khả năng biểu diễn của grid $0.5^\circ$ sẽ không còn nguyên vẹn; đây là phần mất thông tin không thể đảo ngược của mọi phép downsampling.

## 11. Vì sao không lấy cách một điểm?

Cách ngây thơ là giữ mỗi điểm thứ hai theo latitude và longitude. Cách này nhanh nhưng có ba vấn đề:

- các cấu trúc quy mô nhỏ có thể bị alias thành cấu trúc quy mô lớn sai;
- giá trị tại một điểm đơn lẻ không đại diện cho trung bình của ô $0.5^\circ$;
- global mean hoặc area integral của trường có thể thay đổi không kiểm soát.

Bilinear interpolation cũng không phải lựa chọn của dự án. Nó nội suy giá trị tại target point từ các source points gần đó, phù hợp cho nhiều bài toán hiển thị, nhưng không được thiết kế để bảo toàn area integral.

Downloader dùng **first-order conservative area-overlap regridding**. Đây cũng là họ phương pháp WeatherBench2 sử dụng để tạo các bản ERA5 có độ phân giải thấp hơn: xem mỗi source cell có giá trị không đổi và lấy trung bình theo diện tích phần giao với target cell.

## 12. Conservative regridding bằng trực giác

Hãy hình dung target cell $T$ là một ô lớn phủ lên nhiều phần của các source cells $S_i$. Giá trị mới không lấy từ một điểm duy nhất mà là trung bình có trọng số:

$$
x_T=\frac{\sum_i A(T\cap S_i)x_i}{\sum_i A(T\cap S_i)},
$$

trong đó $A(T\cap S_i)$ là diện tích giao giữa target cell và source cell thứ $i$. Source cell phủ target cell nhiều hơn thì đóng góp nhiều hơn.

Vì hai grid được căn tâm tại các bội số của $0.25^\circ$ và $0.5^\circ$, theo longitude một target cell nhận:

$$
\bar x_j=\tfrac14x_{2j-1}+\tfrac12x_{2j}+\tfrac14x_{2j+1}.
$$

Không phải phép lấy trung bình $2\times2$ đơn giản: target cell có tâm trùng một source cell, nên lấy toàn bộ source cell ở giữa và một nửa của mỗi source cell lân cận. Khi chuẩn hóa thành trung bình, trọng số longitude là $0.25, 0.50, 0.25$.

Tại kinh tuyến gốc, chỉ số được wrap theo tính periodic. Vì vậy target cell tâm $0^\circ$ nhận cả đóng góp từ source longitude $359.75^\circ$, thay vì tạo ra một đường nối giả trên bản đồ.

## 13. Trọng số latitude và vai trò của diện tích mặt cầu

Theo latitude, code dựng biên cell từ trung điểm giữa hai cell centres và chặn hai biên ngoài tại $-90^\circ$ và $90^\circ$. Với một dải giao từ $\varphi_a$ đến $\varphi_b$, phần diện tích theo latitude tỉ lệ với

$$
w\propto\sin\varphi_b-\sin\varphi_a.
$$

Các trọng số giao nhau được chuẩn hóa sao cho tổng bằng 1 ở mỗi target latitude. Công thức này xử lý đúng việc các dải latitude gần cực có diện tích nhỏ hơn, thay vì giả định mọi khoảng $0.25^\circ$ đều có cùng diện tích.

Với một trường hằng, output vẫn là đúng hằng số đó. Với trường tổng quát, phương pháp được thiết kế để bảo toàn area-weighted integral đến sai số số học:

$$
\sum_T A_Tx_T\approx\sum_i A_ix_i.
$$

Điều này **không có nghĩa là bảo toàn mọi cực trị địa phương**. Conservative regridding vẫn là phép area averaging, nên peak nhỏ có thể bị làm mượt khi đi từ $0.25^\circ$ xuống $0.5^\circ$.

## 14. Data flow từ archive đến tensor cuối

Luồng biến đổi dữ liệu có thể tóm tắt như sau:

$$
\text{WeatherBench2 Zarr}
\rightarrow\text{giới hạn time; giữ đủ contract 71 kênh}
\rightarrow\text{conservative regridding}
\rightarrow\text{flatten thành channel}
\rightarrow\text{ghi output Zarr}.
$$

Cụ thể:

1. Mở metadata của remote Zarr, chưa tải toàn bộ array vào RAM.
2. Giới hạn date range; lấy cố định sáu surface variables và năm pressure variables trên đủ 13 levels, với cadence 6 giờ.
3. Regrid từng field từ $721\times1440$ xuống $361\times720$.
4. Tách từng pressure level thành một channel riêng và ghép theo channel order cố định.
5. Chuyển về layout $[\text{time},\text{channel},\text{latitude},\text{longitude}]$.
6. Lưu float32 theo chunk $[1,71,361,720]$, tức một timestep hoàn chỉnh trên mỗi chunk logic.

Output variable có tên **state**. Các giá trị vẫn mang đơn vị vật lý được ghi trong channel metadata; notebook này không thực hiện data normalization.

# Phần III — kích thước và kiểm chứng dữ liệu

## 15. Kích thước logic của dataset hiện tại

1995-01-01 đến hết 2020-12-31 có 9,497 ngày, tương đương 37,988 states 6 giờ.

$$\text{shape}=[37{,}988,71,361,720].$$
$$\text{bytes/state}=71\times361\times720\times4=73{,}817{,}280\approx73.82\ \mathrm{MB}.$$
$$\text{total}=2{,}804{,}170{,}832{,}640\ \mathrm{bytes}\approx2804.17\ \mathrm{GB}\approx2611.59\ \mathrm{GiB}.$$

Đây là **kích thước logic chưa nén**, không phải kích thước Zarr thực tế. Cần
đo compression trên sample đại diện trước khi thuê volume sát giới hạn.
Trên source grid 0.25°, 71 kênh tương đương 294.86 MB mỗi state float32 trước
khi tính compression/chunk read amplification. Regridding giảm gần bốn lần
số grid points, giữ nguyên timestep và toàn bộ contract kênh.


## 16. Những kiểm tra bắt buộc trước khi train

### 16.1 Schema và coordinates

- state phải có đúng dimension order $[\text{time},\text{channel},\text{latitude},\text{longitude}]$;
- channel count bằng 71 và channel names khớp tuyệt đối bảng ở mục 7;
- grid bằng $361\times720$; longitude chạy từ $0$ đến $359.5^\circ$; latitude bao gồm cả $90^\circ$ và $-90^\circ$;
- timestamp tăng đều đúng 6 giờ, không trùng và không thiếu.

### 16.2 Giá trị vật lý

- không có NaN hoặc infinity ngoài những trường hợp đã được giải thích;
- units trong metadata đúng với source variable;
- min, max, mean và standard deviation theo channel có quy mô hợp lý;
- surface pressure và mean sea-level pressure không bị tráo;
- geopotential không bị hiểu nhầm là geopotential height;
- specific humidity và relative humidity không bị coi là cùng một đại lượng.

### 16.3 Regridding

- trường hằng vẫn hằng sau regrid;
- area-weighted global integral được bảo toàn trong tolerance;
- không xuất hiện seam bất thường tại $0^\circ/360^\circ$ longitude;
- hai hàng cực không chứa giá trị rác hoặc bị nhân đôi sai.

### 16.4 Tính nhất quán khi resume

Downloader lưu fingerprint của data contract gồm source, dates, cadence, channels và regridding version. Khi resume, fingerprint phải khớp; nếu không, cần dùng output path mới thay vì nối hai dataset có ý nghĩa khác nhau.

## 17. Các nhầm lẫn nên tránh

1. **71 kênh không phải 71 variable names.** Sáu surface variables cộng năm pressure variables trên 13 tầng tạo 71 kênh.
2. **Bộ SFNO 73 kênh không có đủ trong WB2 này.** Thiếu u/v ở 100 m; không nhầm 100 m với tầng 100 hPa.
3. **Giảm từ 0.25° xuống 0.5° không chỉ giảm 2 lần.** Cả hai trục cùng giảm, nên số grid points giảm gần 4 lần.
4. **Conservative không đồng nghĩa với lossless.** Nó ưu tiên bảo toàn area integral; fine-scale details vẫn bị làm mượt.
5. **Grid cell không có diện tích bằng nhau.** Weight theo latitude là bắt buộc cho các phép tính toàn cầu có ý nghĩa vật lý.
6. **Dung lượng logic không bằng dung lượng file nén và cũng không bằng network traffic.** Ba con số này liên quan nhưng không đồng nhất.
7. **Reanalysis không phải ground truth tuyệt đối.** Nó là ước lượng vật lý nhất quán được tạo bằng forecast model và data assimilation.

## 18. Kết luận

Data contract hiện tại của NeuralAtmosphereOperator là:

$$
\boxed{\text{ERA5/WB2, 6-hourly, 71 channels, }0.5^\circ,\ 361\times720}
$$

Mỗi timestep chứa một trạng thái khí quyển toàn cầu gồm gió, pressure/geopotential, temperature và moisture ở bề mặt cùng đủ 13 pressure levels. Dataset nguồn $0.25^\circ$ được đưa về $0.5^\circ$ bằng first-order conservative area-overlap regridding có xử lý đúng diện tích mặt cầu và periodic longitude. Pipeline không còn tùy chọn channel subset; phần bị giảm trong trường dữ liệu là spatial resolution.

Trước khi train, điều quan trọng nhất không phải chỉ nhìn thấy file Zarr tồn tại, mà phải xác nhận schema, channel order, units, cadence, NaN, periodic seam và conservation tests. Khi toàn bộ các kiểm tra này đạt, output mới có thể được xem là dữ liệu đầu vào đáng tin cậy cho model.

## Tài liệu tham khảo

1. Hersbach, H. et al. (2020), [The ERA5 global reanalysis](https://doi.org/10.1002/qj.3803), *Quarterly Journal of the Royal Meteorological Society*.
2. Rasp, S. et al. (2020), [WeatherBench: A benchmark dataset for data-driven weather forecasting](https://arxiv.org/abs/2002.00469), *Journal of Advances in Modeling Earth Systems*.
3. Rasp, S. et al. (2024), [WeatherBench 2: A benchmark for the next generation of data-driven global weather models](https://arxiv.org/abs/2308.15560), *Journal of Advances in Modeling Earth Systems*.
4. [WeatherBench2 Data Guide](https://weatherbench2.readthedocs.io/en/latest/data-guide.html) — archive paths, native grids, cadence, pressure levels và phương pháp tạo lower-resolution datasets.
5. [ECMWF ERA5 documentation](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5) — tổng quan về ERA5 và data assimilation.